# Day 10 — Qwen3-VL-4B · Kaggle QLoRA with checkpoint resume

Notebook này chỉ là entrypoint Kaggle. Pipeline huấn luyện, lưu trạng thái, resume, suy luận và chấm điểm dùng chung trong thư mục `student/` của repository.

Chọn GPU T4, bật Internet, thêm dataset Road Damage có `images/` và `labels-YOLO/`, rồi khai báo Kaggle Secrets `HF_TOKEN` và `HF_REPO_ID` cho các run E1–E4.


## 0. Cấu hình & cài package

Đổi `EXPERIMENT` thành E0, E1, E2, E3 hoặc E4. E4 sẽ chạy cả rank 4 và 64 theo cùng baseline và cùng cấu hình.


In [ ]:
EXPERIMENT = "E1"                  # E0, E1, E2, E3, E4
E4_BASE = "E1"                     # Fixed baseline for the rank ablation
MICRO_BATCH_SIZE = 1                # Effective batch remains 8 with accumulation below
GRAD_ACCUM = 8
MAX_STEPS = 300
CHECKPOINT_EVERY = 25
EVAL_SPLIT = "test"                 # Use val only for debugging
REMOTE_BACKUP_REQUIRED = True
CODE_REF = "feat/day10-kaggle-qlora"


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
REPO = ROOT / "K4-Track-4-Day10-VisionLanguageModel"
REPO_URL = "https://github.com/vbwork2/K4-Track-4-Day10-VisionLanguageModel.git"
if not REPO.is_dir():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", CODE_REF, REPO_URL, str(REPO)], check=True)
else:
    branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=REPO, text=True).strip()
    if branch != CODE_REF:
        raise RuntimeError(f"Expected repository branch {CODE_REF}, found {branch}")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements-colab.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest", "huggingface_hub"], check=True)
sys.path.insert(0, str(REPO))
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests", "student_tests", "--basetemp", str(ROOT / "pytest-tmp")], cwd=REPO, check=True)
GIT_SHA = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
print("Repository:", REPO)
print("Code revision:", GIT_SHA)


## 1. Khởi tạo GPU, Kaggle Secrets, dataset chuẩn của giảng viên

Đính kèm dataset trong tab **Add Input**. Notebook chỉ gọi nguyên bản `scripts/prepare_data.py` để tạo grouped split; không tự chia lại dữ liệu.


In [ ]:
import json
import torch

from PIL import Image
from transformers import AutoProcessor
from student.checkpointing import require_remote_credentials
from student.data import validate_splits
from student.model import MODEL_ID
from scripts.labcommon import read_jsonl

assert torch.cuda.is_available(), "Turn on a Kaggle GPU accelerator before running this notebook."
print("GPU:", torch.cuda.get_device_name(0))

def read_secret(name):
    value = os.environ.get(name)
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

HF_TOKEN = read_secret("HF_TOKEN")
HF_REPO_ID = read_secret("HF_REPO_ID")
if EXPERIMENT != "E0":
    require_remote_credentials(HF_TOKEN, HF_REPO_ID, required=REMOTE_BACKUP_REQUIRED)

input_root = Path("/kaggle/input")
roots = sorted(
    {label_dir.parent for label_dir in input_root.rglob("labels-YOLO")
     if label_dir.is_dir() and (label_dir.parent / "images").is_dir()},
    key=lambda path: len(str(path)),
) if input_root.exists() else []
if not roots:
    raise FileNotFoundError("Attach the Road Damage dataset with images/ and labels-YOLO/ under /kaggle/input.")
DATA_ROOT = roots[0]
DATA_DIR = ROOT / "day10_data"
RUN_ROOT = ROOT / "day10_runs"
if not (DATA_DIR / "splits" / "meta.json").is_file():
    subprocess.run([
        sys.executable, "scripts/prepare_data.py", "--source", str(DATA_ROOT),
        "--out", str(DATA_DIR), "--work", str(ROOT / "day10_prep"),
    ], cwd=REPO, check=True)
DATASET = validate_splits(DATA_DIR, check_expected=True)
ROWS = DATASET["rows"]
DATASET_VERSION = DATASET["meta"]["dataset_version"]
PROCESSOR = AutoProcessor.from_pretrained(MODEL_ID)
print("dataset_version:", DATASET_VERSION)
print("split counts:", DATASET["summary"])
print("Remote backup configured:", bool(HF_TOKEN and HF_REPO_ID))


## 2. Định dạng nhãn 0–1000, run manifest và fingerprint

Chuyển đổi nhãn, prompt và fingerprint được triển khai trong `student/`; đoạn này chạy kiểm tra dữ liệu và overlays.


In [ ]:
import json

from student.labels import objects_to_answer, yolo_to_box
from student.visualization import render_ground_truth_overlays

sample = ROWS["train"][0]
print("Sample answer:", objects_to_answer(sample["objects"], sample["width"], sample["height"]))
print("One box:", yolo_to_box(sample["objects"][0], sample["width"], sample["height"]) if sample["objects"] else [])
LABEL_OVERLAYS = render_ground_truth_overlays(
    ROWS["train"], DATA_DIR, ROOT / "day10_label_overlays", limit=6
)
print("Six label overlays:", LABEL_OVERLAYS)


## 3. Kiểm tra checkpoint trước khi load model

Runner tìm checkpoint tương thích mới nhất trong `/kaggle/working` và private Hugging Face repo. Mismatch hoặc thiếu credential sẽ dừng trước khi train.


In [ ]:
from student.checkpointing import require_remote_credentials
from student.experiments import run_id_for

if EXPERIMENT != "E0":
    require_remote_credentials(HF_TOKEN, HF_REPO_ID, required=REMOTE_BACKUP_REQUIRED)
RUN_ID = run_id_for(EXPERIMENT, E4_BASE if EXPERIMENT == "E4" else None, 4 if EXPERIMENT == "E4" else None)
print("Run identity:", RUN_ID)
print("Checkpoint selection and fingerprint validation happen in run_experiment before loading the model.")


## 4. Qwen3-VL 4-bit: không lượng tử hoá ViT và `lm_head`; E1–E3 target module thực

`student/model.py` nạp NF4/double quant, xác nhận 252 target LLM và 96 target ViT, giữ nguyên `visual` và `lm_head`, rồi mở projector float32 cho E2/E3.


In [ ]:
from student.model import EXPECTED_ARCHITECTURE_COUNTS

if EXPERIMENT in ("E1", "E2", "E3"):
    print("Expected actual Qwen3-VL module counts:", EXPECTED_ARCHITECTURE_COUNTS)
    print("E2/E3 projector state is kept separate from LoRA adapter weights.")
else:
    print("The shared model loader verifies excluded quantization modules and setup before training.")


## 5. Encode ví dụ, mask assistant-only loss; batch vision concat

Notebook chạy prompt-prefix, assistant-label mask, 220 image token và collation qua processor thật trước khi train.


In [ ]:
from PIL import Image

from student.dataset import assert_image_token_count, collate_examples, encode_example
from student.labels import objects_to_answer

sample = ROWS["train"][0]
with Image.open(DATA_DIR / sample["image_path"]) as source:
    image = source.convert("RGB")
answer = objects_to_answer(sample["objects"], sample["width"], sample["height"])
encoded = encode_example(PROCESSOR, image, answer)
image_token_count = assert_image_token_count(PROCESSOR, encoded, expected=220)
print("Image tokens:", image_token_count)
print("Assistant-only answer:", PROCESSOR.tokenizer.decode(encoded["labels"][encoded["labels"] != -100], skip_special_tokens=False))
print("Collated fields:", {key: tuple(value.shape) for key, value in collate_examples([encoded], PROCESSOR).items()})


## 6. Save + restore full training state: adapter, projector, optimizer state và logs

Checkpoint lưu theo step, có integrity manifest; token HF chỉ được đọc từ Kaggle Secrets và không được ghi vào run manifest.


In [ ]:
from student.checkpointing import local_checkpoints

if EXPERIMENT != "E0":
    checkpoints = local_checkpoints(RUN_ROOT / RUN_ID)
    print("Compatible checkpoint search is performed after config fingerprint is built:", len(checkpoints), "local directories")
    print("Remote backup is fail-closed:", REMOTE_BACKUP_REQUIRED)
else:
    print("E0 inference resumes from its JSONL output, one image at a time.")


## 7. Fine-tune: log từng optimizer step, checkpoint mỗi 25 step, resume không lặp

Mỗi lần chạy E1/E2/E3 hoặc cặp E4, nếu phiên Kaggle dừng thì chạy lại notebook với đúng lựa chọn và cấu hình để tiếp tục từ checkpoint remote mới nhất.


In [ ]:
import json

from student.experiments import compare_rank_evidence, run_experiment, run_rank_ablation

if not torch.cuda.is_available():
    raise RuntimeError("Kaggle GPU disappeared before the experiment started.")

def report_step(event):
    if event["step"] == 1 or event["step"] % CHECKPOINT_EVERY == 0:
        print("STEP", event["step"], "loss", round(event["loss"], 5), "peak GB", round(event["peak_gpu_gb"], 2))

if EXPERIMENT == "E4":
    RESULTS = run_rank_ablation(
        REPO, DATA_DIR, RUN_ROOT, base_setup=E4_BASE, split=EVAL_SPLIT,
        micro_batch_size=MICRO_BATCH_SIZE, grad_accum=GRAD_ACCUM,
        max_steps=MAX_STEPS, checkpoint_every=CHECKPOINT_EVERY,
        remote_repo_id=HF_REPO_ID, token=HF_TOKEN,
        remote_required=REMOTE_BACKUP_REQUIRED, on_step=report_step,
    )
    if EVAL_SPLIT == "test":
        E4_COMPARISON = compare_rank_evidence(
            REPO / "report" / "evidence" / f"e4_rank_4_base_{E4_BASE.lower()}.json",
            REPO / "report" / "evidence" / f"e4_rank_64_base_{E4_BASE.lower()}.json",
        )
        print(json.dumps(E4_COMPARISON, ensure_ascii=False, indent=2))
else:
    RESULT = run_experiment(
        EXPERIMENT, REPO, DATA_DIR, RUN_ROOT, split=EVAL_SPLIT,
        micro_batch_size=MICRO_BATCH_SIZE, grad_accum=GRAD_ACCUM,
        max_steps=MAX_STEPS, checkpoint_every=CHECKPOINT_EVERY,
        remote_repo_id=HF_REPO_ID, token=HF_TOKEN,
        remote_required=REMOTE_BACKUP_REQUIRED, evaluate=True, on_step=report_step,
    )
    if EVAL_SPLIT == "test":
        print(json.dumps(RESULT["evidence"], ensure_ascii=False, indent=2))


## 8. Suy luận, score chính thức và evidence

Suy luận JSONL được flush theo từng ảnh. Chỉ tập test đầy đủ mới được chuyển cho `scripts/score.py`; evidence và overlay chỉ được tạo từ dự đoán thật.


In [ ]:
if EXPERIMENT == "E4":
    for result in RESULTS:
        print(result["run_id"], result["evidence"])
else:
    print("Experiment:", RESULT["experiment"])
    print("Prediction JSONL:", RESULT["predictions"])
    print("Evidence:", RESULT.get("evidence"))
    if RESULT.get("evidence"):
        print("First raw outputs:", RESULT["evidence"]["raw_examples"])
        print("Annotated examples:", RESULT["evidence"]["annotated_examples"])


## 9. Đóng gói outputs & cách tiếp tục sau sự cố

Gói tải xuống chứa log, JSONL dự đoán, evidence và hình minh họa; không chứa model, adapter, optimizer hay token.


In [ ]:
import gc
import zipfile

if EXPERIMENT == "E4":
    run_ids = ["e4_rank_4_base_" + E4_BASE.lower(), "e4_rank_64_base_" + E4_BASE.lower()]
else:
    run_ids = [RUN_ID]
ARTIFACT = ROOT / f"{EXPERIMENT.lower()}_logs_evidence.zip"
with zipfile.ZipFile(ARTIFACT, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for run_id in run_ids:
        run_dir = RUN_ROOT / run_id
        for path in run_dir.rglob("*"):
            if path.is_file() and path.parent.name != "checkpoints" and "checkpoints" not in path.parts:
                if path.suffix in (".json", ".jsonl", ".log"):
                    archive.write(path, arcname=f"runs/{run_id}/{path.relative_to(run_dir)}")
        evidence_names = ([run_id] if EXPERIMENT == "E4" else [EXPERIMENT.lower()])
        for evidence_name in evidence_names:
            evidence = REPO / "report" / "evidence" / f"{evidence_name}.json"
            if evidence.is_file():
                archive.write(evidence, arcname=f"evidence/{evidence.name}")
            figures = REPO / "report" / "evidence" / "figures" / evidence_name
            if figures.is_dir():
                for path in figures.glob("*.jpg"):
                    archive.write(path, arcname=f"evidence/figures/{path.name}")
print("Downloadable logs and evidence:", ARTIFACT)
print("Archive excludes all model, adapter, optimizer, and checkpoint files.")


### Checklist trước khi nộp

- GPU T4 đã bật và dataset được attach.
- E1–E4 có private HF repo cùng Secrets hợp lệ.
- E0–E3 và cả rank 4/64 của E4 đã chạy thật trước khi điền `REPORT.md`.
- Đã kiểm tra `dataset_version`, `missing_predictions`, metadata VRAM/thời gian và 6 ảnh overlays.
- GRPO tùy chọn được ghi `NOT_RUN` nếu chưa thực hiện.
